# Rotten Fruits Classification with PyTorch

This notebook provides PyTorch implementations of a simple CNN, a VGG-style CNN, and a VGG-style network with residual blocks.

In [ ]:
# Dependencies
# Run this once in a notebook environment.
# !pip install torch torchvision kaggle scikit-learn

In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

In [ ]:
# #data import

# import os
# os.environ['KAGGLE_USERNAME'] = 'rufaidakassem'
# os.environ['KAGGLE_KEY'] = '14787967'

# import kaggle
# kaggle.api.dataset_download_files('shreyashbhimanwar/rotten-fruits', path='.', unzip=True)

In [ ]:
from pathlib import Path

# Locate the downloaded dataset relative to the repository or current directory.
candidates = [
    Path('/content/dataset'),
    Path.cwd() / 'dataset',
    Path.cwd() / 'rotten-fruits' / 'dataset',
    Path.cwd() / 'notebooks' / 'rotten-fruits' / 'dataset',
]
for candidate in candidates:
    if candidate.is_dir():
        dataset_dir = candidate
        break
else:
    dataset_dir = candidates[0]

train_dir = dataset_dir / 'train'
test_dir = dataset_dir / 'test'
print('Dataset directory:', dataset_dir)
print('Train exists:', train_dir.is_dir())
print('Test exists:', test_dir.is_dir())

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_full = datasets.ImageFolder(str(train_dir), transform=train_transform)
train_full_eval = datasets.ImageFolder(str(train_dir), transform=eval_transform)
raw_test_data = datasets.ImageFolder(str(test_dir), transform=eval_transform)

class_names = train_full.classes
class_to_idx = train_full.class_to_idx
missing_from_test = sorted(set(class_names) - set(raw_test_data.classes))
print('Training classes:', class_names)
print('Test classes:', raw_test_data.classes)
print('Classes missing from test:', missing_from_test)

class RemappedSubset(Dataset):
    def __init__(self, imagefolder, target_class_to_idx):
        self.imagefolder = imagefolder
        self.target_class_to_idx = target_class_to_idx
        self.indices = [
            index
            for index, (_, source_idx) in enumerate(imagefolder.samples)
            if imagefolder.classes[source_idx] in target_class_to_idx
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, source_idx = self.imagefolder[self.indices[index]]
        class_name = self.imagefolder.classes[source_idx]
        target_idx = self.target_class_to_idx[class_name]
        return image, target_idx

# Evaluate only the classes shared by train and test. The original test folder
# has 8 of the 12 training classes, so this prevents label remapping errors.
common_test = RemappedSubset(raw_test_data, class_to_idx)
num_classes = len(class_names)

# Hold out 10 percent of the 12-class training data for validation.
generator = torch.Generator().manual_seed(42)
permuted_indices = torch.randperm(len(train_full), generator=generator).tolist()
validation_size = int(len(train_full) * 0.10)
validation_indices = permuted_indices[:validation_size]
train_indices = permuted_indices[validation_size:]
train_data = Subset(train_full, train_indices)
validation_data = Subset(train_full_eval, validation_indices)

batch_size = 16
loader_workers = 2
train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=loader_workers,
    persistent_workers=True,
    pin_memory=torch.cuda.is_available(),
    )
validation_loader = DataLoader(
    validation_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=loader_workers,
    persistent_workers=True,
    pin_memory=torch.cuda.is_available(),
    )
test_loader = DataLoader(
    common_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=loader_workers,
    persistent_workers=True,
    pin_memory=torch.cuda.is_available(),
    )

print('Training samples:', len(train_data))
print('Validation samples:', len(validation_data))
print('External test samples:', len(common_test))
print('Number of classes:', num_classes)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 54 * 54, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

simple_model = SimpleCNN(num_classes).to(device)
print(simple_model)

In [ ]:
class VGGStyleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

vgg_model = VGGStyleCNN(num_classes).to(device)
print(vgg_model)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class ResidualVGG(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            ResidualBlock(64), ResidualBlock(64),
            nn.Conv2d(64, 128, kernel_size=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block3 = nn.Sequential(
            ResidualBlock(128), ResidualBlock(128), ResidualBlock(128), ResidualBlock(128),
            nn.Conv2d(128, 256, kernel_size=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block4 = nn.Sequential(
            ResidualBlock(256), ResidualBlock(256), ResidualBlock(256), ResidualBlock(256),
            nn.Conv2d(256, 512, kernel_size=1),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.classifier(x)

residual_model = ResidualVGG(num_classes).to(device)
print(residual_model)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0, verbose=False):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

def train_one_epoch(model, loader, criterion, optimizer, epoch, epochs):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    print(f'Epoch {epoch}/{epochs} - loss: {total_loss / total:.4f} - accuracy: {correct / total:.4f}')


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    metrics = {'loss': total_loss / total, 'accuracy': correct / total}
    print(f"Test loss: {metrics['loss']:.4f} - Test accuracy: {metrics['accuracy']:.4f}")
    return metrics


def fit(model, train_loader, validation_loader, epochs=10, learning_rate=1e-4, patience=3, save_path='best_model.pth'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda') if use_amp else None
    history = {'train_loss': [], 'train_accuracy': [], 'validation_loss': [], 'validation_accuracy': []}

    early_stopping = EarlyStopping(patience=patience, verbose=True)
    best_val_loss = float('inf')

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        model.eval()
        validation_loss = 0.0
        validation_correct = 0
        validation_total = 0
        with torch.inference_mode():
            for images, labels in validation_loader:
                images = images.to(device)
                labels = labels.to(device)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                validation_loss += loss.item() * labels.size(0)
                validation_correct += (outputs.argmax(dim=1) == labels).sum().item()
                validation_total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total
        val_loss = validation_loss / validation_total
        val_acc = validation_correct / validation_total
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['validation_loss'].append(val_loss)
        history['validation_accuracy'].append(val_acc)
        print(f'Epoch {epoch}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - validation_loss: {val_loss:.4f} - validation_accuracy: {val_acc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"Saved new best model checkpoint to {save_path}")

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print("Early stopping triggered. Training stopped.")
            break

    return history

## Train Simple CNN

In [ ]:
simple_model = SimpleCNN(num_classes).to(device)
simple_history = fit(simple_model, train_loader, validation_loader, epochs=10, learning_rate=1e-4, save_path='simple_cnn_best.pth')
del simple_model, simple_history
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Train VGG-style CNN

In [ ]:
vgg_model = VGGStyleCNN(num_classes).to(device)
vgg_history = fit(vgg_model, train_loader, validation_loader, epochs=10, learning_rate=1e-4, save_path='vgg_best.pth')
del vgg_model, vgg_history
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Train Residual VGG-style CNN

In [ ]:
residual_model = ResidualVGG(num_classes).to(device)
residual_history = fit(residual_model, train_loader, validation_loader, epochs=10, learning_rate=1e-4, save_path='residual_best.pth')
del residual_model, residual_history
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Model Evaluation on Test Set
Let's evaluate the performance of our models on the test set.

In [ ]:
print("--- Evaluating Simple CNN ---")
simple_model = SimpleCNN(num_classes).to(device)
simple_model.load_state_dict(torch.load('simple_cnn_best.pth'))
evaluate(simple_model, test_loader)

print("\n--- Evaluating VGG CNN ---")
vgg_model = VGGStyleCNN(num_classes).to(device)
vgg_model.load_state_dict(torch.load('vgg_best.pth'))
evaluate(vgg_model, test_loader)

print("\n--- Evaluating Residual VGG ---")
residual_model = ResidualVGG(num_classes).to(device)
residual_model.load_state_dict(torch.load('residual_best.pth'))
evaluate(residual_model, test_loader)